# Pandas 파일별 2 · 05 Final Challenge 숫자로 주장 검토하기

**한 줄 요약** · 준비된 판매 분석표에서 정렬 · groupby · merge · 조건을 조합해 '가장 좋다'는 주장을 기준별 숫자로 검토하고 다음 행동을 제안

- **원본** · `5. pandas/02_최종과제/05_final_challenge.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/04_Pandas` 전체 정리 6 · 7장과 8장 종합 실습 (그룹 비교 · 표 연결 · 처음 보는 판매 데이터)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 정렬 → 앞 n개 → 필요한 열 순서로 큰 거래를 찾고, 동점 · 표본 크기 때문에 단정하면 안 되는 이유를 말한다
2. `groupby` + `agg(["count", "sum", "mean"])`로 '가장 좋다'는 주장을 건수 · 합계 · 평균 기준별로 검토한다
3. `merge(how="left", indicator=True, validate="many_to_one")`로 연결하고 실패 건수 · 금액을 결론에 남긴다
4. 두 조건(`&` · `isin`)으로 점검 목록을 만들고, 숫자 · 코드 · 주의점이 붙은 다음 행동을 제안한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **회의에서 '○○가 가장 좋다'는 주장이 나왔을 때** · 건수 · 합계 · 평균 기준별 검토 (1-2 · 2-3)
- **고객 · 메타 정보를 붙여 분석을 시작할 때** · left merge + 연결 검증 (2-2)
- **운영팀에 넘길 점검 목록을 만들 때** · 조건 두 개 + 정렬 (3-1)
- **분석 결과를 다음 행동으로 바꿀 때** · 근거 숫자 · 코드 · 주의점 (4장)

## 학습 지도

<img src="images/pd2_map.png" alt="Final Challenge 학습 지도" width="763">

**예상 학습 시간** · 개념 30분 + 문제 풀이 45분 (원본 예상 시간: 필수 33 ~ 45분)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>원본 분석용 표 준비 셀 · 문제 안내</td></tr>
<tr><td>1</td><td>큰 거래와 매장 주장</td><td>큰 거래 5건의 공통점과 'Store A가 가장 좋다'는 주장은 숫자로 어떻게 검토하는가?</td></tr>
<tr><td>2</td><td>상품군과 고객군</td><td>집중할 상품군을 고르고, 고객 정보를 붙여 고객군 주장을 검토하려면?</td></tr>
<tr><td>3</td><td>고액 거래 점검</td><td>판매점 · 금액 두 조건으로 점검 목록을 만들고 두 매장을 비교하려면?</td></tr>
<tr><td>4</td><td>종합 실습</td><td>앞의 숫자로 팀장에게 다음 행동 3개를 제안하고, 내 질문 하나에 답하려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

원본의 '분석용 표 준비' 셀을 그대로 실행해 `clean_sales`(596행)를 만듦. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · `pandas` (pandas 3 기준 실행 결과)
- **seed · 장치** · 필요 없음
- **데이터** · `data/unseen.csv` 판매 600행 · `data/unseen_customers.csv` 구매자 180명
- **준비 규칙** · 할인율 결측 → 0 · 상품군 표기 통일 · 날짜 변환 · verified만 · sale_id당 한 행 · 할인 전 · 후 금액

**실행 흐름**

1. `pandas` 불러오기 · 출력 설정
2. 판매 · 고객 CSV 읽기
3. 사본에서 결측 · 문자열 정리
4. verified · 중복 제거 후 할인 전 · 후 금액 계산

In [1]:
# 원본 '분석용 표 준비' 셀 그대로 (원본 파일은 두고 분석용 표를 따로)
# 왜 · 원본 CSV는 그대로 두고 정제한 사본으로만 분석해 언제든 다시 시작할 수 있게 하기 위해
import pandas as pd  # 관례: pd 로 줄여 부름

pd.set_option("display.unicode.east_asian_width", True)  # 한글 열 맞춤
sales_data = pd.read_csv("data/unseen.csv")              # 판매 600행
buyers = pd.read_csv("data/unseen_customers.csv")        # 구매자 180명

# 왜 · 결측 채우기 · 문자열 정리를 원본이 아닌 사본에서 하기 위해
clean_sales = sales_data.copy()                                    # 원본은 확인용으로 남김
clean_sales["coupon_rate"] = clean_sales["coupon_rate"].fillna(0)  # 빈 할인율 = 쿠폰 미사용
clean_sales["item_group"] = clean_sales["item_group"].str.strip().str.upper()  # 표기 통일
clean_sales["sold_at"] = pd.to_datetime(clean_sales["sold_at"])    # 2026/08/05 → 날짜

# verified 거래만 · sale_id 한 개당 한 행
# 왜 · 검증 안 된 거래와 중복 행이 합계를 부풀리지 않게 하기 위해
clean_sales = clean_sales.loc[clean_sales["record_status"] == "verified"]  # 수집 오류 제외
clean_sales = clean_sales.drop_duplicates("sale_id", keep="first").copy()  # ID당 첫 행

# 할인 전 · 후 판매금액
# 왜 · 뒤의 모든 문제가 같은 판매금액 기준을 쓰도록 한 번만 계산해 두기 위해
clean_sales["gross_sales"] = clean_sales["unit_price"] * clean_sales["units"]  # 단가 × 수량
# 할인 반영
clean_sales["final_sales"] = clean_sales["gross_sales"] * (1 - clean_sales["coupon_rate"])
print("원본:", sales_data.shape, "→ 분석용:", clean_sales.shape)               # 600 → 596행
print("sale_id 고유:", clean_sales["sale_id"].is_unique,                       # 준비 검증
      "/ 결측 할인율:", clean_sales["coupon_rate"].isna().sum())

원본: (600, 12) → 분석용: (596, 14)
sale_id 고유: True / 결측 할인율: 0


### 원본 문제

- **진행** · 필수 P01 ~ P06 → 선택 P07 ~ P08 → 도전 P09 · 막히면 힌트 1부터
- **작성 형식** · 내 답 · 확인한 숫자 · 사용한 코드 · 내 해석 · 주의할 점
- **P01** 큰 거래 5건의 공통점 · **P02** "Store A가 가장 성과가 좋다" 검토 · **P03** 집중할 상품군 하나
- **P04** 고객 정보 연결 뒤 고객군 분석을 시작해도 되는지 · **P05** "CORE가 가장 중요한 고객군이다" 검토
- **P06** 팀장에게 다음 행동 3개 · **P07** Store A 고액 거래 점검 목록 · **P08** Store A · B 고액 거래 비교
- **P09** 내 질문 하나 만들고 답하기 (새 기능 없이)

**원본 힌트**

- P01 · 어떤 열로 큰 순서를 만들지 → 정렬 뒤 네 열만 → 상위 5건 안에서 반복 횟수
- P02 · P03 · '성과 · 집중'이 건수인지 금액인지 먼저 나누기 → groupby와 agg
- P04 · 왼쪽에 둘 표와 공통 열 → 고객 ID가 한 행씩인지 확인하는 연결 옵션 → 표시 열로 성공 · 실패
- P05 · 알려진 고객군 표와 연결 실패 행을 따로 · 비어 있는 고객군도 빠지지 않게 묶기
- P07 · P08 · 조건을 괄호로 감싸 `&` · 판매점 두 곳은 `|` · 판매점별 `value_counts()`

---
## 1. 큰 거래와 매장 주장

> **이 장의 흐름**
>
> - 0장에서 분석용 표 596행을 준비
> - P01 큰 거래 5건 → P02 매장별 세 지표 순서

<img src="images/pd2_store.png" alt="매장별 건수 · 총매출 · 평균: 1위가 기준마다 다름" width="892">

**파이프라인 위치** · ⑥ 평가 — 상위 사례 · 그룹 비교

### 1-1. `A` 큰 거래 5건과 공통점 · `sort_values()` · `head()` · `value_counts()` <sub>원본 P01 · 필수</sub>

> 🎯 **`final_sales`가 큰 거래 5건에서 반복되는 공통점은 무엇이고, 왜 단정하면 안 되는가?**

💡 **쉽게 말하면**

- **줄 세우고 앞 5명만** 본 뒤, 그 5명 안에서 상품군 · 판매점을 셈.
- 5건은 596건의 1%도 안 됨.
- 5등 자리에는 같은 금액이 여러 건이라 누가 들어갈지는 정렬 규칙이 정함.

**실행 흐름**

1. 금액 큰 순 (동점은 sale_id 순)
2. 앞 5건 · 네 열
3. 상품군 · 판매점 개수
4. 전체 TECH 비율 · 동점 수

**먼저 예측** · 큰 거래 5건에 가장 많이 나오는 판매점은? — 답은 코드 실행 결과 아래에

In [2]:
# P01 final_sales가 큰 거래 5건 · 필요한 네 열만
# 왜 · 동점이 있어도 같은 5건이 나오도록 두 번째 기준(ID)을 함께 주기 위해
cols = ["sale_id", "item_group", "store", "final_sales"]  # 원본이 지정한 열
# 동점은 ID 순
ranked = clean_sales.sort_values(["final_sales", "sale_id"], ascending=[False, True])
top5 = ranked.head(5)[cols]                               # 앞 5건 · 필요한 열만
print(top5)                                               # 확인한 거래 5건


# 공통점: 상품군 · 판매점이 몇 번씩 나오나
# 왜 · 상위 거래의 공통점을 개수로 확인해 근거 있는 해석을 하기 위해
print("상품군:", top5["item_group"].value_counts().to_dict())                # TECH 4번
print("판매점:", top5["store"].value_counts().to_dict())                     # Store B 4번
# 전체는 다름
print("전체 거래 중 TECH 비율:", round((clean_sales["item_group"] == "TECH").mean(), 3))
print("480,600원 동점 거래:", (clean_sales["final_sales"] == 480600).sum())  # 5등 경계의 동점

        sale_id item_group    store  final_sales
405  SALE000406    FASHION  Store B     565250.0
205  SALE000206       TECH  Store A     480600.0
235  SALE000236       TECH  Store B     480600.0
338  SALE000339       TECH  Store B     480600.0
399  SALE000400       TECH  Store B     480600.0
상품군: {'TECH': 4, 'FASHION': 1}
판매점: {'Store B': 4, 'Store A': 1}
전체 거래 중 TECH 비율: 0.111
480,600원 동점 거래: 6


**결과** · 예측 정답은 Store B — 5건 중 4건

- `SALE000406 FASHION Store B 565250.0` — 1위 거래 · 유일하게 50만 원 이상
- `상품군: {'TECH': 4, 'FASHION': 1}` — 공통점 1: TECH가 4건
- `판매점: {'Store B': 4, 'Store A': 1}` — 공통점 2: Store B가 4건
- `480,600원 동점 거래: 6` — 2 ~ 5등 자리를 6건이 다툼 → 앞 5건은 정렬 규칙에 따라 바뀜

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>sort_values(...).head(5)</code></td><td>정렬 + 앞 5행, 동점 순서는 규칙 따라</td><td>상위 사례 보기</td></tr>
<tr><td><code>sort_values([금액, ID])</code></td><td>동점을 두 번째 기준으로 고정</td><td>결과를 다시 만들 수 있어야 할 때</td></tr>
<tr><td><code>nlargest(5, 열, keep="all")</code></td><td>5등 동점을 모두 포함</td><td>경계 동점을 숨기지 않을 때</td></tr>
<tr><td>상위 5건의 <code>value_counts()</code></td><td>사례 안의 반복</td><td>공통점 찾기 (일반화 X)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 5건의 공통점으로 전체 거래를 단정하지 않음 → 전체 TECH 비율은 0.111뿐

> ✅ **한 줄 정리** · 큰 거래는 정렬 → 앞 n개 → 필요한 열로 보고, 동점과 표본 크기 때문에 공통점을 전체 결론으로 쓰지 않는다.

**딥러닝 코드에서는**

- 가장 확신한 오답 상위 n개를 정렬해 보기 (오류 분석)
- 상위 몇 개 사례로 모델 전체를 판단하지 않기

### 1-2. `A` Store A 주장 검토 · `groupby()` · `agg()` · `idxmax()` <sub>원본 P02 · 필수</sub>

> 🎯 **"Store A가 가장 성과가 좋다"는 어떤 기준에서 맞고, 어떤 기준에서 조심해야 하는가?**

💡 **쉽게 말하면**

- '성과'는 **몇 번 팔았나 · 얼마나 팔았나 · 한 번에 얼마**로 나뉨.
- 세 기준의 1위가 다르면 주장에 기준을 붙여야 함.

**실행 흐름**

1. 매장별 count · sum · mean
2. 지표마다 1위
3. Store A 매출 비중

**먼저 예측** · 매장별 평균 거래금액 1위는 Store A일까? — 답은 코드 실행 결과 아래에

In [3]:
# P02 store별 건수 · 합계 · 평균
# 왜 · 매장을 세 지표로 동시에 비교해 한 지표만 보고 판단하지 않기 위해
metrics = ["count", "sum", "mean"]  # 거래 건수 · 총매출 · 평균
store_summary = clean_sales.groupby("store")["final_sales"].agg(metrics)  # 매장마다 한 줄
print(store_summary.round(0))       # 세 매장 비교


# 지표마다 1위
# 왜 · 지표에 따라 1위가 달라지는지 확인하기 위해
print("1위 → 건수:", store_summary["count"].idxmax(),                       # 기준마다 다른가
      "/ 합계:", store_summary["sum"].idxmax(), "/ 평균:", store_summary["mean"].idxmax())
share_a = store_summary.loc["Store A", "sum"] / store_summary["sum"].sum()  # 전체 매출 중 Store A
print("Store A 매출 비중:", round(share_a, 3))                              # 전체의 몇 %

         count         sum     mean
store                              
Store A    260  12220000.0  47000.0
Store B    150  10500000.0  70000.0
Store C    186   8280000.0  44516.0
1위 → 건수: Store A / 합계: Store A / 평균: Store B
Store A 매출 비중: 0.394


**결과** · 예측 정답은 아님 — 평균은 Store B (70,000원)

- `Store A 260 12220000.0 47000.0` — 건수 · 총매출 1위
- `Store B 150 10500000.0 70000.0` — 평균 거래금액 1위 · 건수는 가장 적음
- `1위 → 건수: Store A / 합계: Store A / 평균: Store B` — 기준마다 1위가 다름
- `Store A 매출 비중: 0.394` — 전체 최종 판매금액의 39.4%

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>count</code></td><td>몇 번 팔았나 (규모)</td><td>운영 부담 · 고객 접점</td></tr>
<tr><td><code>sum</code></td><td>얼마나 팔았나 (규모 × 단가)</td><td>매출 기여</td></tr>
<tr><td><code>mean</code></td><td>한 번에 얼마 (단가)</td><td>고액 거래 성향</td></tr>
<tr><td><code>idxmax()</code></td><td>1위의 이름</td><td>지표별 1위를 바로 비교</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 한 지표만 보면 다른 매장의 강점을 놓침 → 주장에는 기준을 붙여 말하기

> ✅ **한 줄 정리** · 'Store A가 가장 좋다'는 건수 · 총매출 기준에서만 맞고, 한 건당 금액은 Store B가 더 크다.

**딥러닝 코드에서는**

- 모델 비교도 정확도 · F1 · 클래스별 성능처럼 지표마다 1위가 다를 수 있음

---
## 2. 상품군과 고객군

> **이 장의 흐름**
>
> - 1장은 매장 단위였음
> - 같은 방법을 상품군에 적용(P03)
> - 고객 파일을 연결(P04)해 고객군 주장을 검토(P05)

<img src="images/pd2_segment.png" alt="고객군별 건수 · 총매출 · 평균 + 연결 실패" width="892">

**파이프라인 위치** · ① 데이터 준비 · ⑥ 평가 — 표 연결 · 그룹 비교

### 2-1. `A` 집중할 상품군 하나 고르기 · `agg()` · `sort_values()` <sub>원본 P03 · 필수</sub>

> 🎯 **다음 달에 먼저 볼 상품군 하나를 어떤 숫자로 추천하는가?**

💡 **쉽게 말하면**

- '집중'의 기준을 먼저 정하고(합계 · 건수 · 평균), 그 기준의 1위를 추천.
- 다른 기준에서는 순위가 바뀐다는 점을 함께 적음.

**실행 흐름**

1. 상품군별 count · sum · mean
2. 지표마다 높은 순

**먼저 예측** · 총매출 1위 상품군은 건수도 1위일까? — 답은 코드 실행 결과 아래에

In [4]:
# P03 item_group별 건수 · 합계 · 평균
# 왜 · 상품군도 같은 세 지표로 비교해 기준을 통일하기 위해
group_summary = clean_sales.groupby("item_group")["final_sales"].agg(metrics)  # 상품군마다 한 줄
print(group_summary.round(0))                                                  # 다섯 상품군


# 지표마다 1위 · 순위
# 왜 · 지표마다 순위가 어떻게 바뀌는지 한 번에 보기 위해
for col in metrics:                 # 건수 · 합계 · 평균마다
    order = group_summary[col].sort_values(ascending=False).index.tolist()  # 높은 순 이름
    print(f"{col:5} 순위:", order)  # 기준에 따라 순위가 바뀜

            count        sum      mean
item_group                            
FASHION        90  6300000.0   70000.0
FOOD          180  5400000.0   30000.0
LIFESTYLE     150  9750000.0   65000.0
STATIONERY    110  2200000.0   20000.0
TECH           66  7350000.0  111364.0
count 순위: ['FOOD', 'LIFESTYLE', 'STATIONERY', 'FASHION', 'TECH']
sum   순위: ['LIFESTYLE', 'TECH', 'FASHION', 'FOOD', 'STATIONERY']
mean  순위: ['TECH', 'FASHION', 'LIFESTYLE', 'FOOD', 'STATIONERY']


**결과** · 예측 정답은 아님 — 총매출 1위 LIFESTYLE, 건수 1위 FOOD

- `LIFESTYLE 150 9750000.0 65000.0` — 총매출 1위 → 추천 (매출 기여 기준)
- `count 순위: ['FOOD', 'LIFESTYLE', 'STATIONERY', 'FASHION', 'TECH']` — 건수는 FOOD가 1위
- `sum 순위: ['LIFESTYLE', 'TECH', 'FASHION', 'FOOD', 'STATIONERY']` — TECH는 66건뿐인데 총매출 2위
- `mean 순위: ['TECH', 'FASHION', 'LIFESTYLE', 'FOOD', 'STATIONERY']` — 평균은 TECH 1위 (111,364원)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>합계 기준 추천</td><td>매출 기여가 큰 곳</td><td>다음 달 매출 목표 (이 답)</td></tr>
<tr><td>건수 기준</td><td>고객 접점이 많은 곳</td><td>재고 · 운영 관리</td></tr>
<tr><td>평균 기준</td><td>한 번에 많이 사는 곳</td><td>고액 상품 기획</td></tr>
<tr><td><code>agg([...])</code> 한 표</td><td>세 기준을 동시에</td><td>추천 전 순위 변화 확인</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 추천한 LIFESTYLE도 평균은 3위 → '모든 지표 1위'라고 말하지 않기

> ✅ **한 줄 정리** · 상품군 추천은 기준(합계 · 건수 · 평균)을 먼저 정하고, 다른 기준에서 순위가 바뀌는 점을 함께 적는다.

**딥러닝 코드에서는**

- 클래스별 성능을 볼 때도 샘플 수 · 정확도 · 손실 순위가 다를 수 있음

### 2-2. `A` 고객 정보 연결과 검증 · `merge()` · `indicator=` · `validate=` <sub>원본 P04 · 필수</sub>

> 🎯 **판매 표에 고객 정보를 왼쪽 기준으로 붙인 뒤, 고객군 분석을 시작해도 되는지 무엇으로 판단하는가?**

💡 **쉽게 말하면**

- 판매(왼쪽)는 그대로 두고 같은 buyer_id의 고객 정보를 옆에 붙임.
- `validate`는 고객 표가 **한 사람 한 줄**인지 검사.
- `indicator`는 줄마다 찾았는지 도장을 찍음.

<img src="images/pandas_merge.png" alt="공통 Key로 옆에 붙이기 · 못 찾으면 NaN" width="808">

**실행 흐름**

1. left merge + indicator + validate
2. 행 수 전후
3. 성공 · 실패 개수 · 실패 ID
4. 알려진 고객군 비교

**먼저 예측** · 연결 뒤 행 수는 596보다 늘어날까? — 답은 코드 실행 결과 아래에

In [5]:
# P04 고객 정보 왼쪽 연결 (buyer_id 한 행씩인지도 검사)
# 왜 · 거래 수는 그대로 두고 고객 정보만 붙이며, 고객 표 중복으로 행이 늘지 않게 하기 위해
merged = clean_sales.merge(buyers, on="buyer_id", how="left",   # 판매 여러 건 : 구매자 1명
                           indicator=True, validate="many_to_one")
print("연결 전후:", len(clean_sales), "→", len(merged))         # 행 수 유지
print("연결 상태:", merged["_merge"].value_counts().to_dict())  # 성공 · 실패
# 고객 파일에 없음
print("실패 buyer_id:", sorted(merged.loc[merged["_merge"] == "left_only", "buyer_id"].unique()))


# 연결된 거래만 고객군별 비교
# 왜 · 고객 정보가 있는 거래끼리만 공정하게 비교하기 위해
known = merged[merged["_merge"] == "both"]                                  # 성공한 행만
print(known.groupby("buyer_segment")["final_sales"].agg(metrics).round(0))  # 알려진 고객군

연결 전후: 596 → 596
연결 상태: {'both': 588, 'left_only': 8, 'right_only': 0}
실패 buyer_id: ['B9001', 'B9002', 'B9003']
               count         sum     mean
buyer_segment                            
CORE             108   9720000.0  90000.0
NEW              280  10080000.0  36000.0
REGULAR          200  10400000.0  52000.0


**결과** · 예측 정답은 그대로 596 — 고객 표가 한 사람 한 줄 (validate 통과)

- `연결 전후: 596 → 596` — 행 유지
- `연결 상태: {'both': 588, 'left_only': 8, 'right_only': 0}` — 성공 588 · 실패 8
- `실패 buyer_id: ['B9001', 'B9002', 'B9003']` — 고객 파일에 없는 ID 3개
- `CORE 108 9720000.0 90000.0` — 알려진 고객군 비교 시작 가능 (실패 8건은 따로 보고)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>how="left"</code></td><td>판매 전부 유지 · 못 찾으면 NaN</td><td>분석 기준 표를 지킬 때 (기본 선택)</td></tr>
<tr><td><code>how="inner"</code></td><td>연결된 것만 · 실패 8건이 <strong>사라짐</strong></td><td>실패 건수를 따로 적을 때만</td></tr>
<tr><td><code>indicator=True</code></td><td><code>_merge</code> 열: both · left_only</td><td>성공 · 실패 세기</td></tr>
<tr><td><code>validate="many_to_one"</code></td><td>고객 ID가 중복이면 에러</td><td>행이 몰래 늘어나는 것 방지</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 코드가 실행됐다고 연결 성공이 아님 → 행 수 · `_merge` 개수를 항상 확인

> ✅ **한 줄 정리** · left merge에 `indicator` · `validate`를 붙여 행 수 유지 · 연결 실패를 확인한 뒤 고객군 분석을 시작한다.

**딥러닝 코드에서는**

- 예측 결과(left)에 정답 표를 붙여 채점 · `left_only`는 정답이 없는 샘플

### 2-3. `A` CORE 주장 검토 · `groupby(dropna=False)` <sub>원본 P05 · 필수</sub>

> 🎯 **"CORE가 가장 중요한 고객군이다"를 기준이 드러나는 문장으로 고치면?**

💡 **쉽게 말하면**

- '중요'가 규모(건수 · 합계)인지, 한 건당 금액인지 나눔.
- 연결 실패 8건도 한 줄로 함께 보여 줌.

**실행 흐름**

1. dropna=False로 NaN 그룹까지
2. 알려진 고객군만 1위
3. 연결 실패 비중

**먼저 예측** · CORE는 어떤 기준에서 1위일까? — 답은 코드 실행 결과 아래에

In [6]:
# P05 연결 실패까지 한 표로 (NaN 그룹 포함)
# 왜 · 고객 정보를 못 찾은 거래가 결과에서 조용히 빠지지 않게 하기 위해
seg = merged.groupby("buyer_segment", dropna=False)["final_sales"].agg(metrics)  # NaN = 연결 실패
print(seg.round(0))                                                              # 네 줄


# CORE 주장 검토: 기준마다 1위 · 실패 규모
# 왜 · 주장이 연결 실패 거래를 빼서 생긴 착시가 아닌지 확인하기 위해
named = seg[seg.index.notna()]                                               # 알려진 고객군만
top = named.idxmax()                                                         # 열마다 1위 고객군
print("1위 → 건수:", top["count"], "/ 합계:", top["sum"], "/ 평균:", top["mean"])  # 기준별
fail = merged["buyer_segment"].isna()                                        # 연결 실패 행
money = merged.loc[fail, "final_sales"].sum() / merged["final_sales"].sum()  # 금액 비중
print("연결 실패 비중:", round(fail.mean(), 3), "/ 금액 비중:", round(money, 3))  # 결론에 남길 숫자

               count         sum      mean
buyer_segment                             
CORE             108   9720000.0   90000.0
NEW              280  10080000.0   36000.0
REGULAR          200  10400000.0   52000.0
NaN                8    800000.0  100000.0
1위 → 건수: NEW / 합계: REGULAR / 평균: CORE
연결 실패 비중: 0.013 / 금액 비중: 0.026


**결과** · 예측 정답은 평균 거래금액 (90,000원) — 건수는 NEW, 합계는 REGULAR

- `NaN 8 800000.0 100000.0` — 연결 실패 8건 · 80만 원
- `1위 → 건수: NEW / 합계: REGULAR / 평균: CORE` — CORE는 한 건당 금액에서만 1위
- `연결 실패 비중: 0.013 / 금액 비중: 0.026` — 거래의 1.3% · 금액의 2.6%는 고객군을 모름
- 고친 문장 · CORE는 한 건당 거래금액이 가장 큰 고객군(90,000원)이고, 총매출은 REGULAR가 가장 큼 (연결 실패 8건 제외)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>groupby("열")</code></td><td>NaN 그룹 <strong>제외</strong></td><td>알려진 그룹끼리 비교</td></tr>
<tr><td><code>groupby("열", dropna=False)</code></td><td>NaN도 한 그룹</td><td>연결 실패 규모를 보고할 때</td></tr>
<tr><td>건수 · 합계</td><td>규모</td><td>얼마나 많이</td></tr>
<tr><td>평균</td><td>한 건당 크기</td><td>한 번에 얼마</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 연결 실패 거래의 평균(100,000원)이 CORE보다 큼 → 실패 거래의 고객군이 밝혀지면 결론이 바뀔 수 있음을 적기

> ✅ **한 줄 정리** · 고객군 주장은 기준을 밝힌 문장으로 고치고, 연결 실패 건수 · 금액을 결론에 함께 남긴다.

**딥러닝 코드에서는**

- 라벨이 없는 샘플을 조용히 빼지 말고 개수를 보고

---
## 3. 고액 거래 점검

> **이 장의 흐름**
>
> - 1 · 2장은 전체 거래 비교
> - 선택 문제 P07 · P08: 200,000원 이상 고액 거래만 따로 점검

<img src="images/pd2_highvalue.png" alt="200,000원 이상 거래: Store A 2건 vs Store B 8건" width="637">

**파이프라인 위치** · ③ 전처리 · ⑥ 평가 — 조건 필터 · 점검 목록

### 3-1. `B` Store A 고액 거래 점검 목록 · `&` · `sort_values()` <sub>원본 P07 · 선택</sub>

> 🎯 **Store A에서 200,000원 이상 거래를 큰 순으로 정리하면 몇 건인가?**

💡 **쉽게 말하면**

- 판매점 조건 **그리고** 금액 조건 → 큰 순 → 운영팀이 볼 열만.

**실행 흐름**

1. is_a · is_big
2. & 로 고르기 → 큰 순
3. 상위 5건 (실제는 2건)

**먼저 예측** · Store A의 200,000원 이상 거래는 5건이 넘을까? — 답은 코드 실행 결과 아래에

In [7]:
# P07 Store A에서 final_sales 200,000원 이상
# 왜 · 두 조건을 각각 이름 있는 mask로 만들어 읽기 쉽게 하기 위해
is_a = clean_sales["store"] == "Store A"                                          # 판매점 조건
is_big = clean_sales["final_sales"] >= 200000                                     # 금액 조건
check_a = clean_sales[is_a & is_big].sort_values("final_sales", ascending=False)  # 두 조건 → 큰 순
print("Store A 고액 거래:", len(check_a))                                         # 점검 대상 수
print(check_a.head(5)[["sale_id", "item_group", "units", "coupon_rate", "final_sales"]])  # 상위 5건

Store A 고액 거래: 2
        sale_id item_group  units  coupon_rate  final_sales
205  SALE000206       TECH      6         0.10     480600.0
283  SALE000284    FASHION      4         0.05     452200.0


**결과** · 예측 정답은 아님 — 2건뿐

- `Store A 고액 거래: 2` — 점검 대상 2건
- `205 SALE000206 TECH 6 0.10 480600.0` — 먼저 볼 거래: TECH 6개 · 10% 쿠폰
- `283 SALE000284 FASHION 4 0.05 452200.0` — 두 번째: FASHION 4개 · 5% 쿠폰

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df[(a) &amp; (b)]</code></td><td>두 조건 모두</td><td>판매점 + 금액 (이 답)</td></tr>
<tr><td>조건 → <code>sort_values()</code></td><td>고른 뒤 큰 순</td><td>점검 순서</td></tr>
<tr><td><code>head(5)</code></td><td>최대 5행 (적으면 있는 만큼)</td><td>상위 n건 요청</td></tr>
<tr><td><code>len()</code></td><td>점검 건수</td><td>목록 크기 보고</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `head(5)`를 요청해도 2건만 나옴 → 건수를 먼저 말하기

> ✅ **한 줄 정리** · 점검 목록은 조건마다 괄호를 친 `&`로 고르고 큰 순으로 정렬한 뒤 건수부터 보고한다.

**딥러닝 코드에서는**

- `errors = df[(df.pred != df.y) & (df.conf > 0.9)]` — 확신한 오답만 점검

### 3-2. `B` Store A · B 고액 거래 비교 · `isin()` · `groupby()` <sub>원본 P08 · 선택</sub>

> 🎯 **Store A와 B의 200,000원 이상 거래만 비교하면 어느 판매점을 먼저 봐야 하는가?**

💡 **쉽게 말하면**

- 판매점 두 곳은 `isin`(또는 `|`), 금액 조건은 `&`.
- 고른 표를 판매점별로 묶어 세 지표와 상품군을 봄.

**실행 흐름**

1. isin + 금액 조건
2. 판매점별 count · sum · mean
3. 판매점별 상품군 개수

**먼저 예측** · 고액 거래 수는 Store A와 B 중 어디가 많을까? — 답은 코드 실행 결과 아래에

In [8]:
# P08 Store A · B의 200,000원 이상 거래만
# 왜 · 두 매장의 큰 거래만 골라 같은 기준으로 비교하기 위해
big_ab = clean_sales[clean_sales["store"].isin(["Store A", "Store B"]) & is_big]  # 두 매장 | → isin
print(big_ab.groupby("store")["final_sales"].agg(metrics).round(0))  # 매장별 건수 · 합 · 평균


# 매장 안에서 자주 보이는 상품군
# 왜 · 매장마다 큰 거래를 이끄는 상품군을 확인하기 위해
for store, part in big_ab.groupby("store"):                    # 매장마다 나눠서
    print(store, part["item_group"].value_counts().to_dict())  # 상품군 반복 확인

         count        sum      mean
store                              
Store A      2   932800.0  466400.0
Store B      8  3872650.0  484081.0
Store A {'TECH': 1, 'FASHION': 1}
Store B {'TECH': 5, 'FASHION': 3}


**결과** · 예측 정답은 Store B — 8건 (A는 2건)

- `Store A 2 932800.0 466400.0` — 고액 2건 · 합 932,800원
- `Store B 8 3872650.0 484081.0` — 고액 8건 · 합 3,872,650원 → B를 먼저 점검
- `Store B {'TECH': 5, 'FASHION': 3}` — B의 고액 거래는 TECH · FASHION뿐
- 주의 · 고액 거래만 본 비교 · 전체 건수 · 총매출은 Store A가 1위 (1-2)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>isin(["Store A", "Store B"])</code></td><td><code>&#124;</code>를 두 번 쓴 것과 같음</td><td>같은 열의 여러 값 (기본 선택)</td></tr>
<tr><td><code>(a) &#124; (b)</code></td><td>또는</td><td>열이 서로 다른 조건</td></tr>
<tr><td><code>groupby("store")</code> 반복</td><td>매장마다 나눠 처리</td><td>매장별 상품군 확인</td></tr>
<tr><td>전체 거래 비교 (1-2)</td><td>모든 금액</td><td>평소 성과</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 고액 거래만 대상으로 한 비교 → 전체 성과 결론과 섞지 않기

> ✅ **한 줄 정리** · 두 매장 비교는 `isin`과 금액 조건을 `&`로 묶어 고르고, 고액 거래만 본 결과임을 함께 적는다.

**딥러닝 코드에서는**

- 특정 조건 부분집합의 성능(예: 작은 물체만)을 볼 때도 대상 범위를 함께 적음

---
## 4. 종합 실습 · 다음 행동 3개와 내 질문

> **이 장의 흐름**
>
> - 필수 P06: 큰 거래 · 판매점 또는 상품군 · 고객군에서 하나씩 제안
> - 도전 P09: 새 기능 없이 결제 수단(payment) 질문 하나

<img src="images/pd2_practice.png" alt="제안 3개 = 근거 숫자 + 다음 행동 + 주의" width="733">

**파이프라인 위치** · ⑥ 평가 → 보고

> 🎯 **각 제안을 숫자 하나 · 그 숫자를 만든 코드 · 누가 무엇을 할지 · 주의점으로 쓰려면?**

💡 **쉽게 말하면**

- 앞 표를 다시 계산하지 않고 필요한 숫자만 모음.
- 관찰에서 끝내지 않고 다음 행동으로 씀.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>큰 거래 5건 중 Store B 4건</td><td>1-1 <code>top5</code></td></tr>
<tr><td>2</td><td>매장 기준별 1위 · Store B 평균</td><td>1-2 <code>store_summary</code></td></tr>
<tr><td>3</td><td>연결 실패 8건 · CORE 평균</td><td>2-2 · 2-3 <code>merged</code> · <code>named</code></td></tr>
<tr><td>4</td><td>결제 수단별 평균 · 쿠폰 사용 비율 (P09)</td><td>groupby + 이름 붙인 agg</td></tr>
</tbody></table>

**먼저 예측** · 결제 수단 중 평균 거래금액이 가장 큰 것은? — 답은 코드 실행 결과 아래에

In [9]:
# P06 · 제안 세 개에 쓸 숫자를 다시 모음 (앞 표 재사용)
# 왜 · 제안의 근거 숫자를 새로 계산하지 않고 앞 결과에서 가져와 일관성을 지키기 위해
a_first = (store_summary[["count", "sum"]].idxmax() == "Store A").all()  # 두 지표 모두 A인가
facts = {                                                                # 제안마다 쓸 근거 숫자
    "큰 거래 5건 중 Store B": int((top5["store"] == "Store B").sum()),   # 큰 거래
    "Store A 건수 · 총매출 1위": a_first,                                # 판매점
    "Store B 평균 (원)": int(store_summary.loc["Store B", "mean"]),      # 판매점
    "연결 실패 거래": int(merged["buyer_segment"].isna().sum()),         # 고객군
    "CORE 평균 (원)": int(named.loc["CORE", "mean"]),                    # 고객군
}
for k, v in facts.items():                                               # 제안마다
    print(f"{k}: {v}")                                                   # 근거 숫자 출력

큰 거래 5건 중 Store B: 4
Store A 건수 · 총매출 1위: True
Store B 평균 (원): 70000
연결 실패 거래: 8
CORE 평균 (원): 90000


**실행 흐름**

1. 결제 수단별로 묶기
2. 거래 수 · 평균 거래금액 · 쿠폰 사용 비율 계산
3. 표로 출력

In [10]:
# P09 내 질문: 결제 수단마다 평균 거래금액 · 쿠폰 사용 비율은?
# 왜 · 스스로 정한 질문도 같은 정제 데이터와 같은 방법으로 답하기 위해
pay = clean_sales.groupby("payment").agg(                       # 결제 수단별 여러 지표
    count=("final_sales", "size"),                              # 거래 건수
    mean_sales=("final_sales", "mean"),                         # 평균 금액
    coupon_used=("coupon_rate", lambda r: (r > 0).mean()),      # 쿠폰을 쓴 비율
)
print(pay.round(3).sort_values("mean_sales", ascending=False))  # 평균 금액 높은 순

               count  mean_sales  coupon_used
payment                                      
Bank Transfer    100   61404.000        0.370
KakaoPay         130   55308.462        0.408
Card             280   49867.143        0.386
Other             86   43101.163        0.395


**결과** · 예측 정답은 Bank Transfer (61,404원)

- `큰 거래 5건 중 Store B: 4` — 제안 1 근거 · Store B 고액 TECH 거래 재고 · 응대 점검 (운영팀) · 주의: 5건뿐
- `Store A 건수 · 총매출 1위: True` — 제안 2 근거 · 매장 평가는 건수 · 총매출 · 평균을 함께 보고
- `Store B 평균 (원): 70000` — Store B는 건당 금액 1위 → Store B의 거래 수를 늘리는 방안 검토 · 주의: 원인은 미확인
- `연결 실패 거래: 8` — 제안 3 근거 · 2-2의 실패 ID 3개 고객 정보 보완 (데이터팀) 뒤 고객군 결론 확정
- `CORE 평균 (원): 90000` — CORE 대상 고액 기획은 평균 기준 근거 · 주의: 총매출 1위는 REGULAR
- `Bank Transfer 100 61404.000 0.370` — P09 · 평균 1위지만 쿠폰 사용 비율은 가장 낮음 (0.370)
- `Other 86 43101.163 0.395` — 평균이 가장 작은 결제 수단

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>관찰만 쓰기</td><td>"B가 고액이 많다"</td><td>보고서 초안</td></tr>
<tr><td>제안 = 숫자 + 행동 + 주의</td><td>누가 무엇을 확인할지까지</td><td>팀장 보고 (이 답)</td></tr>
<tr><td>이름 붙인 <code>agg(이름=(열, 함수))</code></td><td>열마다 다른 계산 · 결과 열 이름 지정</td><td>P09처럼 지표가 여럿일 때</td></tr>
<tr><td><code>lambda r: (r &gt; 0).mean()</code></td><td>조건 비율</td><td>쿠폰 사용 비율</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 숫자가 '원인'을 말하지는 않음 → 제안은 '확인 · 검토'로 쓰고 단정하지 않기

> ✅ **한 줄 정리** · 다음 행동은 근거 숫자 하나와 그 숫자를 만든 코드, 누가 무엇을 확인할지, 주의점을 함께 적어 제안한다.

**딥러닝 코드에서는**

- 실험 보고도 같은 형식 — 지표 숫자 · 재현 코드 · 다음 실험 · 한계

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 큰 거래 · 동점</td><td>앞 5건 중 TECH 4 · Store B 4, 5등 경계 동점 6건</td><td>1-1</td></tr>
<tr><td>2. 기준별 주장 검토</td><td>Store A는 건수 · 총매출 1위 · 평균은 Store B 70,000원</td><td>1-2 · 2-1</td></tr>
<tr><td>3. 연결 검증</td><td>596 → 596 · 성공 588 · 실패 8 · CORE는 평균만 1위</td><td>2-2 · 2-3</td></tr>
<tr><td>4. 점검 · 제안</td><td>Store A 고액 2건 vs B 8건 · 제안 3개 · Bank Transfer 평균 1위</td><td>3장 · 4</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>상위 사례</td><td>정렬 → 앞 n → 열 · 동점 주의</td><td>확신한 오답 상위 n개</td></tr>
<tr><td><code>A</code></td><td>기준별 비교</td><td><code>agg(["count", "sum", "mean"])</code></td><td>지표별 모델 비교</td></tr>
<tr><td><code>A</code></td><td>연결 검증</td><td>left + <code>indicator</code> + <code>validate</code></td><td>예측 + 정답 붙이기</td></tr>
<tr><td><code>A</code></td><td>NaN 그룹</td><td><code>groupby(dropna=False)</code></td><td>라벨 없는 샘플 수 보고</td></tr>
<tr><td><code>B</code></td><td>점검 목록</td><td>괄호 + <code>&amp;</code> · <code>isin</code></td><td>조건 부분집합 성능</td></tr>
<tr><td><code>B</code></td><td>제안 형식</td><td>숫자 · 코드 · 행동 · 주의</td><td>실험 보고서</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 큰 거래 5건의 공통점을 전체 결론으로 쓰면 안 되는 이유는?
- **1-2** · "Store A가 가장 좋다"가 맞는 기준은?
- **2-1** · 상품군 하나를 추천할 때 먼저 정할 것은?
- **2-2** · 연결 뒤 고객군 분석을 시작해도 되는지 무엇으로 판단하는가?
- **2-3** · CORE 주장을 고치면?
- **3-1** · 점검 목록을 만드는 순서는?
- **3-2** · 고액 거래 비교 결과를 쓸 때 주의는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `clean_sales`에서 Store C의 평균 `final_sales`를 반올림 정수로 출력
2. `units`가 5개 이상인 거래 수 출력

In [11]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. P07의 금액 기준을 100,000원으로 낮추면 Store A 점검 대상은 몇 건?
2. P04를 `how="inner"`로 연결하면 행 수와 전체 `final_sales` 합계는?
3. P01을 `nlargest(5, "final_sales", keep="all")`로 바꾸면 몇 행이 나올까?

In [12]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 5건뿐이고 5등 경계에 동점 6건 · 전체 TECH 비율은 0.111
- **1-2** · 건수 · 총매출 (평균은 Store B)
- **2-1** · 기준 (합계 · 건수 · 평균)
- **2-2** · 행 수 유지 · `_merge` 성공 · 실패 · validate 통과
- **2-3** · 한 건당 금액 1위 · 총매출은 REGULAR · 연결 실패 8건 제외
- **3-1** · 조건(괄호 + `&amp;`) → 정렬 → 필요한 열 · 건수 보고
- **3-2** · 고액 거래만 대상 · 전체 성과와 섞지 않기

#### 코드 문제 답

```python
print(round(clean_sales.loc[clean_sales["store"] == "Store C", "final_sales"].mean()))  # 44516
print((clean_sales["units"] >= 5).sum())  # 26
```

#### 변형 문제 답

*문제 → 답*

- **1** · 24건 — `(is_a &amp; (clean_sales["final_sales"] &gt;= 100000)).sum()`
- **2** · 588행 · 31000000 - 800000 = 30200000원 (실패 8건이 빠짐)
- **3** · 7행 — 1위 1건 + 480,600원 동점 6건을 모두 포함

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) "Store A가 가장 좋다"를 그대로 맞다고 씀</td><td>총매출 하나만 봄</td><td>건수 · 합계 · 평균을 함께 보고 기준을 붙임</td><td>이 주장은 어떤 기준에서 맞는가?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. '가장 성과가 좋다'는 주장을 데이터로 어떻게 검토하나요?**
> - 성과를 건수, 총매출, 평균처럼 측정 가능한 기준으로 나누고 groupby와 agg로 한 표에 계산합니다.
> - 기준마다 1위가 다르면 주장에 기준을 붙여 고쳐 씁니다.
> - 이번 데이터에서 Store A는 건수와 총매출 1위였지만 평균은 Store B가 1위였습니다.

**Q2. 고객 정보를 연결한 뒤 무엇을 확인하나요?**
> - left merge에 validate를 걸어 고객 표가 한 사람 한 줄인지 확인하고, 행 수가 그대로인지 봅니다.
> - indicator로 연결 실패 건수를 세고, 실패 건수와 금액을 결론에 함께 남깁니다.

**Q3. 분석 결과를 다음 행동으로 바꾸는 방법은요?**
> - 제안마다 근거 숫자 하나와 그 숫자를 만든 코드, 누가 무엇을 확인할지, 주의점을 함께 씁니다.
> - 숫자는 원인을 말해 주지 않으므로 단정 대신 확인이나 검토 행동으로 제안합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>.sort_values([..], ascending=[..])</code> · <code>.head()</code></td><td>DataFrame 메서드</td><td>동점 기준까지 정렬 · 앞 n행</td><td>1-1 <code>clean_sales.sort_values(["final_sales", "sale_id"], ...)</code></td></tr>
<tr><td><code>.value_counts()</code></td><td>Series 메서드</td><td>사례 안의 반복</td><td>1-1 <code>top5["item_group"].value_counts()</code></td></tr>
<tr><td><code>.groupby().agg([..])</code></td><td>DataFrame · GroupBy</td><td>그룹마다 건수 · 합계 · 평균</td><td>1-2 <code>groupby("store")["final_sales"].agg(metrics)</code></td></tr>
<tr><td><code>.idxmax()</code></td><td>Series 메서드</td><td>1위의 이름</td><td>1-2 <code>store_summary["count"].idxmax()</code></td></tr>
<tr><td><code>.merge(how="left", indicator=, validate=)</code></td><td>DataFrame 메서드</td><td>연결 + 검증</td><td>2-2 <code>clean_sales.merge(buyers, on="buyer_id", ...)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>groupby(dropna=False)</code></td><td>DataFrame 메서드</td><td>NaN 그룹 포함</td><td>2-3 <code>merged.groupby("buyer_segment", dropna=False)</code></td></tr>
<tr><td><code>&amp;</code> · <code>.isin()</code></td><td>pandas 연산자 · Series 메서드</td><td>두 조건 · 여러 값</td><td>3-2 <code>clean_sales["store"].isin([...]) &amp; is_big</code></td></tr>
<tr><td><code>agg(이름=(열, 함수))</code></td><td>GroupBy 메서드</td><td>이름 붙인 여러 지표</td><td>4 <code>coupon_used=("coupon_rate", lambda r: ...)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>주장에는 기준(건수 · 합계 · 평균)을 붙이기</td><td>지표 하나로 '가장 좋다' 단정</td></tr>
<tr><td>merge 뒤 행 수 · <code>_merge</code> 확인</td><td><code>inner</code>로 연결 실패를 조용히 삭제</td></tr>
<tr><td>동점은 두 번째 기준으로 순서 고정</td><td>앞 5건의 공통점을 전체 결론으로</td></tr>
<tr><td>제안 = 숫자 + 코드 + 행동 + 주의</td><td>숫자로 원인을 단정</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>count</code> vs <code>sum</code> vs <code>mean</code></td><td>몇 번 vs 얼마나 vs 한 번에 얼마</td><td>질문에 맞는 기준</td></tr>
<tr><td><code>how="left"</code> vs <code>"inner"</code></td><td>판매 유지 vs 연결된 것만</td><td>보통 left + indicator</td></tr>
<tr><td><code>groupby()</code> vs <code>dropna=False</code></td><td>NaN 제외 vs 포함</td><td>실패 규모 보고는 포함</td></tr>
<tr><td><code>head(5)</code> vs <code>nlargest(keep="all")</code></td><td>정확히 5행 vs 동점 모두</td><td>경계 동점 확인</td></tr>
<tr><td><code>isin</code> vs <code>&#124;</code></td><td>같은 열 여러 값 vs 다른 조건</td><td>같은 열이면 <code>isin</code></td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · P02 주장을 다른 문장으로 고쳐 쓰기</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>원본 P01 ~ P08을 빈 노트북에서 다시 풀고 숫자 비교</td><td>45분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 전체 정리 6 · 7 · 8장 ✅ 줄만 읽기</td><td>15분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `05_final_challenge.ipynb`의 분석용 표 준비 셀 · P01 ~ P09 · 힌트 1 ~ 3
- 원본 문제 칸은 모두 `# TODO`였음 → 9문제를 직접 풀고 실행 결과로 '내 답 · 숫자 · 코드 · 해석 · 주의'를 채움
- 필수 P01 · P02 → 1장, P03 · P04 · P05 → 2장, 선택 P07 · P08 → 3장, P06 · 도전 P09 → 4장 종합 실습
- 원본의 접힌 힌트(details 태그)는 0장 '원본 힌트'로 · P01 정렬은 동점 때문에 `sale_id`를 두 번째 기준으로 추가
- 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>분석용 표 준비 셀</td><td>할인율 0 · 표기 통일 · 날짜 · verified · ID당 한 행 · 금액</td><td>0장</td></tr>
<tr><td>P01 큰 거래 5건 (필수 · 난이도 2)</td><td>정렬 · 공통점 · 단정 금지</td><td>1-1</td></tr>
<tr><td>P02 Store A 주장 (필수 · 난이도 3)</td><td>store별 지표 2개 이상</td><td>1-2</td></tr>
<tr><td>P03 상품군 하나 (필수 · 난이도 3)</td><td>item_group 지표 · 추천</td><td>2-1</td></tr>
<tr><td>P04 고객군 분석 시작 판단 (필수 · 난이도 3)</td><td>left 연결 · 성공/실패 · 한 행 조건</td><td>2-2</td></tr>
<tr><td>P05 CORE 주장 (필수 · 난이도 4)</td><td>지표 2개 이상 · 연결 실패 · 고쳐 쓰기</td><td>2-3</td></tr>
<tr><td>P06 다음 행동 3개 (필수 · 난이도 4)</td><td>큰 거래 · 판매점/상품군 · 고객군</td><td>4 종합 실습</td></tr>
<tr><td>P07 Store A 고액 점검 (선택 · 난이도 3)</td><td>&amp; · 정렬 · 상위 5건</td><td>3-1</td></tr>
<tr><td>P08 Store A · B 고액 비교 (선택 · 난이도 3)</td><td></td><td>· 판매점별 지표 · value_counts</td></tr>
<tr><td>P09 내 질문 (도전 · 난이도 4)</td><td>payment · units · coupon_rate 중 선택</td><td>4 종합 실습</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>마무리</td><td>준비된 표 확인 → 선택 → 지표 결정 → 주장 검토 → 행동 · 주의</td><td>그래서 오늘 무엇을 배웠는가?</td></tr>
</tbody></table>

### 전체 코드

1 ~ 4장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 상위 거래 5건과 공통점
2. 1-2 매장별 세 지표 · 1위

In [13]:
# ---------- 1-1 ----------
# P01 final_sales가 큰 거래 5건 · 필요한 네 열만
# 왜 · 동점이 있어도 같은 5건이 나오도록 두 번째 기준(ID)을 함께 주기 위해
cols = ["sale_id", "item_group", "store", "final_sales"]  # 원본이 지정한 열
# 동점은 ID 순
ranked = clean_sales.sort_values(["final_sales", "sale_id"], ascending=[False, True])
top5 = ranked.head(5)[cols]                               # 앞 5건 · 필요한 열만
print(top5)                                               # 확인한 거래 5건


# 공통점: 상품군 · 판매점이 몇 번씩 나오나
# 왜 · 상위 거래의 공통점을 개수로 확인해 근거 있는 해석을 하기 위해
print("상품군:", top5["item_group"].value_counts().to_dict())                # TECH 4번
print("판매점:", top5["store"].value_counts().to_dict())                     # Store B 4번
# 전체는 다름
print("전체 거래 중 TECH 비율:", round((clean_sales["item_group"] == "TECH").mean(), 3))
print("480,600원 동점 거래:", (clean_sales["final_sales"] == 480600).sum())  # 5등 경계의 동점


# ---------- 1-2 ----------
# P02 store별 건수 · 합계 · 평균
# 왜 · 매장을 세 지표로 동시에 비교해 한 지표만 보고 판단하지 않기 위해
metrics = ["count", "sum", "mean"]  # 거래 건수 · 총매출 · 평균
store_summary = clean_sales.groupby("store")["final_sales"].agg(metrics)  # 매장마다 한 줄
print(store_summary.round(0))       # 세 매장 비교


# 지표마다 1위
# 왜 · 지표에 따라 1위가 달라지는지 확인하기 위해
print("1위 → 건수:", store_summary["count"].idxmax(),                       # 기준마다 다른가
      "/ 합계:", store_summary["sum"].idxmax(), "/ 평균:", store_summary["mean"].idxmax())
share_a = store_summary.loc["Store A", "sum"] / store_summary["sum"].sum()  # 전체 매출 중 Store A
print("Store A 매출 비중:", round(share_a, 3))                              # 전체의 몇 %

        sale_id item_group    store  final_sales
405  SALE000406    FASHION  Store B     565250.0
205  SALE000206       TECH  Store A     480600.0
235  SALE000236       TECH  Store B     480600.0
338  SALE000339       TECH  Store B     480600.0
399  SALE000400       TECH  Store B     480600.0
상품군: {'TECH': 4, 'FASHION': 1}
판매점: {'Store B': 4, 'Store A': 1}
전체 거래 중 TECH 비율: 0.111
480,600원 동점 거래: 6
         count         sum     mean
store                              
Store A    260  12220000.0  47000.0
Store B    150  10500000.0  70000.0
Store C    186   8280000.0  44516.0
1위 → 건수: Store A / 합계: Store A / 평균: Store B
Store A 매출 비중: 0.394


**실행 흐름**

1. 2-1 상품군별 세 지표
2. 2-2 고객 정보 왼쪽 연결 · 검사
3. 2-3 연결 실패까지 포함한 비교

In [14]:
# ---------- 2-1 ----------
# P03 item_group별 건수 · 합계 · 평균
# 왜 · 상품군도 같은 세 지표로 비교해 기준을 통일하기 위해
group_summary = clean_sales.groupby("item_group")["final_sales"].agg(metrics)  # 상품군마다 한 줄
print(group_summary.round(0))                                                  # 다섯 상품군


# 지표마다 1위 · 순위
# 왜 · 지표마다 순위가 어떻게 바뀌는지 한 번에 보기 위해
for col in metrics:                 # 건수 · 합계 · 평균마다
    order = group_summary[col].sort_values(ascending=False).index.tolist()  # 높은 순 이름
    print(f"{col:5} 순위:", order)  # 기준에 따라 순위가 바뀜


# ---------- 2-2 ----------
# P04 고객 정보 왼쪽 연결 (buyer_id 한 행씩인지도 검사)
# 왜 · 거래 수는 그대로 두고 고객 정보만 붙이며, 고객 표 중복으로 행이 늘지 않게 하기 위해
merged = clean_sales.merge(buyers, on="buyer_id", how="left",   # 판매 여러 건 : 구매자 1명
                           indicator=True, validate="many_to_one")
print("연결 전후:", len(clean_sales), "→", len(merged))         # 행 수 유지
print("연결 상태:", merged["_merge"].value_counts().to_dict())  # 성공 · 실패
# 고객 파일에 없음
print("실패 buyer_id:", sorted(merged.loc[merged["_merge"] == "left_only", "buyer_id"].unique()))


# 연결된 거래만 고객군별 비교
# 왜 · 고객 정보가 있는 거래끼리만 공정하게 비교하기 위해
known = merged[merged["_merge"] == "both"]                                  # 성공한 행만
print(known.groupby("buyer_segment")["final_sales"].agg(metrics).round(0))  # 알려진 고객군


# ---------- 2-3 ----------
# P05 연결 실패까지 한 표로 (NaN 그룹 포함)
# 왜 · 고객 정보를 못 찾은 거래가 결과에서 조용히 빠지지 않게 하기 위해
seg = merged.groupby("buyer_segment", dropna=False)["final_sales"].agg(metrics)  # NaN = 연결 실패
print(seg.round(0))                                                              # 네 줄


# CORE 주장 검토: 기준마다 1위 · 실패 규모
# 왜 · 주장이 연결 실패 거래를 빼서 생긴 착시가 아닌지 확인하기 위해
named = seg[seg.index.notna()]                                               # 알려진 고객군만
top = named.idxmax()                                                         # 열마다 1위 고객군
print("1위 → 건수:", top["count"], "/ 합계:", top["sum"], "/ 평균:", top["mean"])  # 기준별
fail = merged["buyer_segment"].isna()                                        # 연결 실패 행
money = merged.loc[fail, "final_sales"].sum() / merged["final_sales"].sum()  # 금액 비중
print("연결 실패 비중:", round(fail.mean(), 3), "/ 금액 비중:", round(money, 3))  # 결론에 남길 숫자

            count        sum      mean
item_group                            
FASHION        90  6300000.0   70000.0
FOOD          180  5400000.0   30000.0
LIFESTYLE     150  9750000.0   65000.0
STATIONERY    110  2200000.0   20000.0
TECH           66  7350000.0  111364.0
count 순위: ['FOOD', 'LIFESTYLE', 'STATIONERY', 'FASHION', 'TECH']
sum   순위: ['LIFESTYLE', 'TECH', 'FASHION', 'FOOD', 'STATIONERY']
mean  순위: ['TECH', 'FASHION', 'LIFESTYLE', 'FOOD', 'STATIONERY']
연결 전후: 596 → 596
연결 상태: {'both': 588, 'left_only': 8, 'right_only': 0}
실패 buyer_id: ['B9001', 'B9002', 'B9003']
               count         sum     mean
buyer_segment                            
CORE             108   9720000.0  90000.0
NEW              280  10080000.0  36000.0
REGULAR          200  10400000.0  52000.0
               count         sum      mean
buyer_segment                             
CORE             108   9720000.0   90000.0
NEW              280  10080000.0   36000.0
REGULAR          200  10400000.0   520

**실행 흐름**

1. 3-1 Store A 큰 거래
2. 3-2 Store A · B 큰 거래 비교 · 상품군

In [15]:
# ---------- 3-1 ----------
# P07 Store A에서 final_sales 200,000원 이상
# 왜 · 두 조건을 각각 이름 있는 mask로 만들어 읽기 쉽게 하기 위해
is_a = clean_sales["store"] == "Store A"                                          # 판매점 조건
is_big = clean_sales["final_sales"] >= 200000                                     # 금액 조건
check_a = clean_sales[is_a & is_big].sort_values("final_sales", ascending=False)  # 두 조건 → 큰 순
print("Store A 고액 거래:", len(check_a))                                         # 점검 대상 수
print(check_a.head(5)[["sale_id", "item_group", "units", "coupon_rate", "final_sales"]])  # 상위 5건


# ---------- 3-2 ----------
# P08 Store A · B의 200,000원 이상 거래만
# 왜 · 두 매장의 큰 거래만 골라 같은 기준으로 비교하기 위해
big_ab = clean_sales[clean_sales["store"].isin(["Store A", "Store B"]) & is_big]  # 두 매장 | → isin
print(big_ab.groupby("store")["final_sales"].agg(metrics).round(0))  # 매장별 건수 · 합 · 평균


# 매장 안에서 자주 보이는 상품군
# 왜 · 매장마다 큰 거래를 이끄는 상품군을 확인하기 위해
for store, part in big_ab.groupby("store"):                    # 매장마다 나눠서
    print(store, part["item_group"].value_counts().to_dict())  # 상품군 반복 확인

Store A 고액 거래: 2
        sale_id item_group  units  coupon_rate  final_sales
205  SALE000206       TECH      6         0.10     480600.0
283  SALE000284    FASHION      4         0.05     452200.0
         count        sum      mean
store                              
Store A      2   932800.0  466400.0
Store B      8  3872650.0  484081.0
Store A {'TECH': 1, 'FASHION': 1}
Store B {'TECH': 5, 'FASHION': 3}


**실행 흐름**

1. 제안에 쓸 숫자 모으기
2. 내 질문(결제 수단) 분석

In [16]:
# ---------- 4. 종합 실습 ----------
# P06 · 제안 세 개에 쓸 숫자를 다시 모음 (앞 표 재사용)
# 왜 · 제안의 근거 숫자를 새로 계산하지 않고 앞 결과에서 가져와 일관성을 지키기 위해
a_first = (store_summary[["count", "sum"]].idxmax() == "Store A").all()  # 두 지표 모두 A인가
facts = {                                                                # 제안마다 쓸 근거 숫자
    "큰 거래 5건 중 Store B": int((top5["store"] == "Store B").sum()),   # 큰 거래
    "Store A 건수 · 총매출 1위": a_first,                                # 판매점
    "Store B 평균 (원)": int(store_summary.loc["Store B", "mean"]),      # 판매점
    "연결 실패 거래": int(merged["buyer_segment"].isna().sum()),         # 고객군
    "CORE 평균 (원)": int(named.loc["CORE", "mean"]),                    # 고객군
}
for k, v in facts.items():                                               # 제안마다
    print(f"{k}: {v}")                                                   # 근거 숫자 출력


# P09 내 질문: 결제 수단마다 평균 거래금액 · 쿠폰 사용 비율은?
# 왜 · 스스로 정한 질문도 같은 정제 데이터와 같은 방법으로 답하기 위해
pay = clean_sales.groupby("payment").agg(                       # 결제 수단별 여러 지표
    count=("final_sales", "size"),                              # 거래 건수
    mean_sales=("final_sales", "mean"),                         # 평균 금액
    coupon_used=("coupon_rate", lambda r: (r > 0).mean()),      # 쿠폰을 쓴 비율
)
print(pay.round(3).sort_values("mean_sales", ascending=False))  # 평균 금액 높은 순

큰 거래 5건 중 Store B: 4
Store A 건수 · 총매출 1위: True
Store B 평균 (원): 70000
연결 실패 거래: 8
CORE 평균 (원): 90000
               count  mean_sales  coupon_used
payment                                      
Bank Transfer    100   61404.000        0.370
KakaoPay         130   55308.462        0.408
Card             280   49867.143        0.386
Other             86   43101.163        0.395
